# Large Action Model (LAM)
Decision Transformer (Chen et al., 2021), Gato (Reed et al., 2022), RT-2 (Brohan et al., 2023), and "agent" models that operate apps and computers

**Idea:** a Transformer that takes an **instruction + an observation of the environment** and outputs an **action**, which is then executed. The environment changes, the model observes again, and so on: a closed loop. Trained here by **behavior cloning**: imitate an expert's actions.

**Toy environment:** a 7x7 grid with walls, the agent, and 3 objects (a color + a shape). The instruction is `go to the <color> <shape>`. Actions: `up`, `down`, `left`, `right`. Success = standing on the named object within 30 steps. The model has to **ground** the words in the grid (which cell is "the blue key"?) and **plan** a path around walls.

### LAM vs LLM (SLM/RLM)
| | LLM | LAM |
|---|---|---|
| Input | text tokens | instruction tokens **+ observation tokens** (grid cells, screenshots, robot camera) |
| Output | next text token | next **action** (move, click, API call, motor command) |
| Loop | generate text | act -> environment changes -> observe -> act ... |
| Training data | text | expert trajectories (demonstrations), then RL |
| Evaluation | loss / answer accuracy | **task success rate** in the environment |
| Failure mode | wrong text | one wrong action shifts the state (errors compound) |

Shape flow: `[CLS] + instruction (B, 5) + grid cells (B, 49) -> embed (B, 55, D) -> encoder x L -> CLS (B, D) -> action logits (B, 4)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

import random
from collections import deque
random.seed(42)

## 1. Environment and expert
Cells: `0` empty, `1` wall, `2` agent, `3..14` objects (4 colors x 3 shapes). Objects are passable; walls and the border block moves (the agent stays put).

The **expert** knows the shortest path: BFS from the target gives the distance of every cell, and the expert moves to the neighbor with the smallest distance.

In [ ]:
SIZE = 7
COLORS = ["red", "green", "blue", "yellow"]
SHAPES = ["ball", "key", "box"]
ACTIONS = ["up", "down", "left", "right"]
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
EMPTY, WALL, AGENT = 0, 1, 2
obj_id = lambda color, shape: 3 + 3 * color + shape
num_cell_types = 3 + len(COLORS) * len(SHAPES)       # 15

WORDS = ["go", "to", "the"] + COLORS + SHAPES
wtoi = {w: i for i, w in enumerate(WORDS)}

def bfs_distances(grid, target):
    # distance of every cell to target (walls = inf)
    dist = [[math.inf] * SIZE for _ in range(SIZE)]
    dist[target[0]][target[1]] = 0
    q = deque([target])
    while q:
        r, c = q.popleft()
        for dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            if 0 <= nr < SIZE and 0 <= nc < SIZE and grid[nr][nc] != WALL and dist[nr][nc] == math.inf:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist


class GridEnv:
    def __init__(self, num_walls=8):
        while True:
            self.grid = [[EMPTY] * SIZE for _ in range(SIZE)]
            cells = random.sample([(r, c) for r in range(SIZE) for c in range(SIZE)], num_walls + 4)
            for r, c in cells[4:]:
                self.grid[r][c] = WALL
            self.agent = cells[0]
            objs = random.sample([(co, sh) for co in range(len(COLORS)) for sh in range(len(SHAPES))], 3)
            for (r, c), (co, sh) in zip(cells[1:4], objs):
                self.grid[r][c] = obj_id(co, sh)
            k = random.randrange(3)
            self.target, (co, sh) = cells[1 + k], objs[k]
            self.instruction = f"go to the {COLORS[co]} {SHAPES[sh]}"
            self.dist = bfs_distances(self.grid, self.target)
            if self.dist[self.agent[0]][self.agent[1]] < math.inf:   # reachable
                break

    def observe(self):
        # -> instruction ids (5,), cells (49,)
        cells = [row[:] for row in self.grid]
        cells[self.agent[0]][self.agent[1]] = AGENT
        instr = torch.tensor([wtoi[w] for w in self.instruction.split()])
        return instr, torch.tensor(cells).flatten()

    def step(self, action):
        r, c = self.agent[0] + MOVES[action][0], self.agent[1] + MOVES[action][1]
        if 0 <= r < SIZE and 0 <= c < SIZE and self.grid[r][c] != WALL:
            self.agent = (r, c)
        return self.agent == self.target                               # done / success

    def expert_action(self):
        r, c = self.agent
        best = min(range(4), key=lambda a: self.dist[r + MOVES[a][0]][c + MOVES[a][1]]
                   if 0 <= r + MOVES[a][0] < SIZE and 0 <= c + MOVES[a][1] < SIZE else math.inf)
        return best

env = GridEnv()
print(env.instruction, "| agent", env.agent, "| target", env.target, "| expert:", ACTIONS[env.expert_action()])

## 2. Demonstrations
Roll out the expert in many random episodes and record every `(instruction, observation, action)` step. This is the LAM's training data, like recorded human demonstrations of a task.

In [ ]:
def collect(num_episodes):
    instrs, obs, acts = [], [], []
    for _ in range(num_episodes):
        env = GridEnv()
        done = False
        while not done:
            i, o = env.observe()
            a = env.expert_action()
            instrs.append(i); obs.append(o); acts.append(a)
            done = env.step(a)
    return torch.stack(instrs), torch.stack(obs), torch.tensor(acts)

train_instr, train_obs, train_act = collect(30000)
print("steps:", len(train_act), "| instr", tuple(train_instr.shape), "| obs", tuple(train_obs.shape))
print("action counts:", torch.bincount(train_act).tolist())

## 3. Action model
One token sequence per step: `[CLS]`, the 5 instruction words, the 49 grid cells. Each token = its embedding (word or cell type) + a learned position embedding, so the model knows where each cell is. A Pre-LN Transformer encoder (the ViT block) lets every token attend to every other: words find the matching object, the object and agent cells find each other and the walls between them. The action is read from `[CLS]`, as in ViT.

### Formula: Policy
$$\pi_\theta(a \mid \ell, s) = \text{softmax}\left(W_a\, h_{CLS}\right), \qquad h = \text{Encoder}\left([e_{CLS};\ e(\ell_1..\ell_5);\ e(s_1..s_{49})] + E_{pos}\right)$$
- $\ell$: instruction words, $s$: the 49 grid cells (observation)
- $e(\cdot)$: word / cell-type embeddings ($D = 128$), $E_{pos} \in \mathbb{R}^{55 \times D}$
- Encoder: 4 Pre-LN blocks, no mask (attention and block formulas as in `vit.ipynb`)
- $W_a \in \mathbb{R}^{4 \times D}$: action head, one logit per action

In [ ]:
class EncoderBlock(nn.Module):
    # (B, T, D) -> (B, T, D); same as ViT
    def __init__(self, dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_ratio * dim), nn.GELU(), nn.Linear(mlp_ratio * dim, dim))

    def forward(self, x):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]   # (B, T, D)
        x = x + self.mlp(self.ln2(x))                       # (B, T, D)
        return x


class ActionModel(nn.Module):
    # instr (B, 5), cells (B, 49) -> action logits (B, 4)
    def __init__(self, dim=128, depth=4, num_heads=4):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.word_embed = nn.Embedding(len(WORDS), dim)
        self.cell_embed = nn.Embedding(num_cell_types, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + 5 + SIZE * SIZE, dim))
        self.blocks = nn.Sequential(*[EncoderBlock(dim, num_heads) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, len(ACTIONS))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, instr, cells):
        B = instr.shape[0]
        x = torch.cat([self.cls.expand(B, -1, -1),           # (B, 1, D)
                       self.word_embed(instr),               # (B, 5, D)
                       self.cell_embed(cells)], dim=1)       # (B, 49, D) -> (B, 55, D)
        x = self.blocks(x + self.pos_embed)                  # (B, 55, D)
        return self.head(self.ln(x[:, 0]))                   # CLS -> (B, 4)

model = ActionModel().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(train_instr[:2].to(device), train_obs[:2].to(device)).shape)

## 4. Behavior cloning
Supervised learning on expert actions: classify "which action would the expert take here?".

### Formula: Behavior cloning loss
$$\mathcal{L}_{BC} = -\frac{1}{|\mathcal{D}|}\sum_{(\ell, s, a^*) \in \mathcal{D}} \log \pi_\theta(a^* \mid \ell, s)$$
- $\mathcal{D}$: recorded expert steps
- $a^*$: the expert's action in state $s$ for instruction $\ell$
- plain cross-entropy over the 4 actions

In [ ]:
epochs = 5
batch_size = 256
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
n = len(train_act)

for epoch in range(epochs):
    perm = torch.randperm(n)
    total_loss = correct = 0
    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        instr, obs, act = train_instr[idx].to(device), train_obs[idx].to(device), train_act[idx].to(device)
        logits = model(instr, obs)                          # (B, 4)
        loss = F.cross_entropy(logits, act)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)
        correct += (logits.argmax(1) == act).sum().item()
    print(f"epoch {epoch+1} | loss {total_loss / n:.4f} | action acc {correct / n:.4f}")

## 5. Closed-loop evaluation
The real test for an action model: let it **act** in new environments, feeding its own actions back in. Action accuracy on expert states is not the same thing, because one mistake puts the agent in a state the expert never visited.

### Formula: Success rate
$$\text{SR} = \frac{1}{N}\sum_{j=1}^{N} \mathbb{1}\left[\exists\, t \le T_{max}: s^{(j)}_t = \text{goal}^{(j)}\right]$$
- $N$: number of new test episodes (500), $T_{max} = 30$ steps

### Formula: Compounding errors (why closed loop is harder)
$$\text{cost}(\pi_\theta) \le \text{cost}(\pi^*) + O(\epsilon\, T^2)$$
- $\epsilon$: per-step error rate on expert states, $T$: episode length (Ross & Bagnell, 2010)
- errors grow with $T^2$, not $T$, because each mistake leads to unfamiliar states; fixes: DAgger (ask the expert about the model's own states) or RL

In [ ]:
@torch.no_grad()
def rollout(model, env, max_steps=30):
    model.eval()
    path = [env.agent]
    for _ in range(max_steps):
        instr, cells = env.observe()
        action = model(instr[None].to(device), cells[None].to(device)).argmax(-1).item()  # (1, 4) -> action
        done = env.step(action)
        path.append(env.agent)
        if done:
            return True, path
    return False, path

random.seed(0)
results = [rollout(model, GridEnv()) for _ in range(500)]
successes = [ok for ok, _ in results]
print(f"success rate: {sum(successes) / len(successes):.3f}")

## 6. Watch an episode

In [ ]:
from matplotlib.colors import ListedColormap

def show(env, path, ok, ax):
    img = [[0 if v == EMPTY else 1 if v == WALL else 2 + (v - 3) // 3 for v in row] for row in env.grid]  # color by object color
    cmap = ListedColormap(["white", "black", "red", "green", "blue", "gold"])
    ax.imshow(img, cmap=cmap, vmin=0, vmax=5)
    for r in range(SIZE):
        for c in range(SIZE):
            if env.grid[r][c] >= 3:
                ax.text(c, r, SHAPES[(env.grid[r][c] - 3) % 3], ha="center", va="center", fontsize=7)
    ys, xs = zip(*path)
    ax.plot(xs, ys, "m-o", markersize=3)
    ax.set_title(f"{env.instruction}\n{'success' if ok else 'fail'} in {len(path) - 1} steps", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax in axes:
    env = GridEnv()
    ok, path = rollout(model, env)
    show(env, path, ok, ax)
plt.tight_layout(); plt.show()

## Notes
- Expect high action accuracy and a slightly lower closed-loop success rate: the gap is the compounding-error effect.
- Real LAMs are the same recipe at scale: observations are screenshots / camera images (ViT tokens) or web page DOMs, actions are clicks, keystrokes, API calls or robot joint commands, and a pretrained LLM/VLM supplies the language understanding (RT-2, OpenVLA, computer-use agents).
- After behavior cloning they are usually improved with RL on task success, the same "try, get rewarded, update" loop as GRPO in `rlm.ipynb`.